## DATA TRANSFORMTIONS

In [0]:
from pyspark.sql.functions import col, upper, when, current_timestamp

# 1. Setup Paths
storage_account = "rgstoragecustomer"
source_path = f"abfss://raw-landing@{storage_account}.dfs.core.windows.net/customer_reviews_large"
target_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/customer_reviews_cleaned"

def load_bronze_data(path):
    print(f"Reading data from {path}...")
    return spark.read.format("parquet").load(path)

def clean_data(df):
    # Nano-step: Clean the text and handle nulls
    return df.withColumn("review_text", upper(col("review_text"))) \
             .withColumn("is_high_rating", when(col("rating") >= 4, True).otherwise(False)) \
             .withColumn("ingestion_timestamp", current_timestamp()) \
             .filter(col("review_text").isNotNull())

# Execution
raw_df = load_bronze_data(source_path)
cleaned_df = clean_data(raw_df)

# 2. Write to Silver using Delta Format
cleaned_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(target_path)

print("Silver Layer Created Successfully!")

In [0]:
df = spark.read.format("delta").load(target_path).count()

In [0]:
display(df)